# Dark Matter Halo Two-Point Correlation Function (2PCF)

This notebook computes the two-point correlation function ξ(r) for **dark matter halos** using the Corrfunc library with periodic boundary conditions.

- **DM Halos**: Represented by central galaxies (is_central=1) from galaxies.hdf5
- **Subvolumes**: 1024 independent random realizations of the same 542³ Mpc³/h³ box
- **Periodic Box**: Corrfunc applies periodic boundary conditions with box size = 542 Mpc/h
- **Max Scale**: rmax < boxsize/2 ≈ 271 Mpc/h to avoid double-counting pairs

## Load Libraries

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from galform_analysis.utils.matplotlib_config import register_matplotlib_setconfig
import importlib
from galform_analysis.config import get_base_dir
import galform_analysis.analysis.correlation.dm_correlation as dm_corr

# Reload module to get latest fixes
dm_corr = importlib.reload(dm_corr)
dm_correlation_given_redshift_and_subvolume = dm_corr.dm_correlation_given_redshift_and_subvolume
avg_dm_correlation_given_subvolume_and_redshifts = dm_corr.avg_dm_correlation_given_subvolume_and_redshifts

base_dir = get_base_dir()
register_matplotlib_setconfig(mpl)
mpl.setconfig()

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 11

## Dark Matter Halo 2PCF from Merger Trees

In [ ]:
# Compute DM correlation from galaxies.hdf5 (halo catalog)
example_snapshot = 'iz207'
example_ivol = 100
snap_path = str(base_dir / example_snapshot)

# Define bin edges
rbins = np.logspace(np.log10(0.1), np.log10(50.0), 20)

# Optional: apply minimum host halo mass cut (mhhalo is the host halo mass)
mhhalo_min = 1e11  # Msun (host halo mass)

dm_result = dm_correlation_given_redshift_and_subvolume(
    iz_path=snap_path,
    ivol=example_ivol,
    rbins=rbins,
    nthreads=4,
    mhhalo_min=mhhalo_min,
)

if dm_result is None:
    raise RuntimeError("DM correlation computation failed")

z = dm_result.attrs.get('z')
nhalo = dm_result.attrs.get('ngal')  # Actually number of halos
boxsize = dm_result.attrs.get('boxsize')
V_ivol = dm_result.attrs.get('V_ivol')

print(f"Redshift: {z:.4f}")
print(f"N_halos: {nhalo}")
print(f"Boxsize: {boxsize:.1f} Mpc/h")
print(f"V_ivol: {V_ivol:.3e} Mpc³/h³")


In [ ]:
r = dm_result['r']
xi_dm = dm_result['xi']
z = dm_result.attrs.get('z')
nhalo = dm_result.attrs.get('ngal')
boxsize = dm_result.attrs.get('boxsize')

# Print data table
print(f"\n{'r [Mpc/h]':>12}  {'ξ_DM(r)':>12}")
print("-" * 26)
for i in range(0, len(r), 2):
    if np.isfinite(xi_dm[i]):
        print(f"{r[i]:12.2f}  {xi_dm[i]:12.6f}")

plt.figure(figsize=(8, 6))
mask = (xi_dm > 0) & np.isfinite(xi_dm)
plt.loglog(r[mask], xi_dm[mask], 'o-', color='darkred', linewidth=2, markersize=6, label=f'z={z:.2f}')
plt.xlabel('r [Mpc/h]', fontsize=12)
plt.ylabel('ξ(r)', fontsize=12)
title = "Dark Matter Halo 2PCF"
title += f"\n(z={z:.2f}, N_halo={nhalo}, M_host>={mhhalo_min:.0e} Msun)"
plt.title(title, fontsize=11)
if boxsize is not None:
    plt.text(0.02, 0.02, f"L={boxsize:.1f} Mpc/h", transform=plt.gca().transAxes, fontsize=9)
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
iz_nums = [100, 120, 142, 176, 207]
example_ivol = 100

results = []
for iz_num in iz_nums:
    iz_path = str(base_dir / f'iz{iz_num}')
    res = dm_correlation_given_redshift_and_subvolume(
        iz_path=iz_path,
        ivol=example_ivol,
        rbins=rbins,
        nthreads=4,
        mhhalo_min=mhhalo_min,
    )
    if res is not None:
        res.attrs['iz'] = f'iz{iz_num}'
        results.append(res)
    else:
        print(f"iz{iz_num}: failed")

plt.figure(figsize=(8, 6))
for res in results:
    r_vals = res['r']
    xi_vals = res['xi']
    z_val = res.attrs.get('z')
    mask = (xi_vals > 0) & np.isfinite(xi_vals)
    plt.loglog(r_vals[mask], xi_vals[mask], 'o-', markersize=4, label=f"z={z_val:.2f}")
plt.xlabel('r [Mpc/h]', fontsize=12)
plt.ylabel('ξ(r)', fontsize=12)
plt.title(f'DM Halo 2PCF Evolution (ivol={example_ivol}, M_host>={mhhalo_min:.0e} Msun)', fontsize=11)
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()